# Incident Response Runbook: Collection - Input Capture

**Tactic:** Collection
**Technique:** Input Capture (T1056)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Input Capture activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os
from splunk_data_collector import SplunkDataCollector
from crowdstrike_response import CrowdStrikeResponse
from iris_integration import create_iris_case, update_iris_case
from misp_integration import search_iocs, create_event
from shuffle_integration import trigger_workflow

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Collection',
    'technique': 'Input Capture',
    'severity': 'HIGH'
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")

# Query Splunk for input capture indicators
print(f"\n[ACTION] Querying Splunk for input capture activities...")
input_capture_queries = [
    'index=* "GetAsyncKeyState" OR "SetWindowsHookEx" OR "keylogger" OR "input capture" | stats count by host, user, process',
    'index=* "WH_KEYBOARD_LL" OR "WH_MOUSE_LL" OR "keyboard hook" | stats count by host, user',
    'index=* source="Sysmon" EventCode=10 OR EventCode=11 | search "user32.dll" OR "kernel32.dll" | stats count by host, user, process',
    'index=* "SetWindowsHookExA" OR "SetWindowsHookExW" | stats count by host, user, dll_path',
    'index=* "raw input" OR "RegisterRawInputDevices" | stats count by host, user'
]

suspicious_activities = []
for query in input_capture_queries:
    try:
        results = splunk.search(query, earliest_time="-24h@h", latest_time="now")
        if results and len(results) > 0:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} suspicious input capture events")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for keylogger behavior
print(f"\n[ACTION] Analyzing endpoint behavior with CrowdStrike...")
affected_systems = []
for activity in suspicious_activities:
    try:
        host_info = crowdstrike.get_host_info(activity.get('host', ''))
        if host_info:
            affected_systems.append(host_info)
            print(f"   Host {activity.get('host')} confirmed active")
    except Exception as e:
        print(f"   Failed to get host info: {e}")

# Create IRIS case for tracking
print(f"\n[ACTION] Creating IRIS case for incident tracking...")
try:
    iris_case = create_iris_case(
        title=f"Input Capture Incident - {alert_data['alert_id']}",
        description=f"Detected input capture/keylogger activities affecting {len(affected_systems)} systems",
        severity="High",
        tags=["collection", "input-capture", "keylogger", "T1056"]
    )
    iris_case_id = iris_case.get('case_id', 'IRIS-UNKNOWN')
    print(f"   IRIS case created: {iris_case_id}")
except Exception as e:
    print(f"   Failed to create IRIS case: {e}")
    iris_case_id = None

# Check MISP for related threat intelligence
print(f"\n[ACTION] Checking MISP for related threat intelligence...")
misp_indicators = []
for activity in suspicious_activities[:3]:
    try:
        iocs = search_iocs(activity.get('process', ''), type='filename')
        if iocs:
            misp_indicators.extend(iocs)
            print(f"   Found {len(iocs)} related IOCs in MISP")
    except Exception as e:
        print(f"   MISP search failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious input capture activities detected")
print(f"  - {len(affected_systems)} systems affected")
print(f"  - {len(misp_indicators)} related threat intelligence indicators")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_systems = []
terminated_processes = []
blocked_network = []

# Isolate affected systems immediately
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'success':
            isolated_systems.append(system['hostname'])
            print(f"   Isolated system: {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Terminate keylogger processes
print(f"\n[ACTION] Terminating keylogger and input capture processes...")
for activity in suspicious_activities:
    try:
        processes = crowdstrike.get_processes(activity.get('host', ''), activity.get('user', ''))
        for proc in processes:
            if any(keyword in proc.get('name', '').lower() for keyword in ['keylogger', 'hook', 'capture', 'spy', 'logger']):
                termination_result = crowdstrike.terminate_process(activity.get('host', ''), proc['pid'])
                if termination_result.get('status') == 'success':
                    terminated_processes.append(f"{activity.get('host')}:{proc['name']}")
                    print(f"   Terminated process: {proc['name']} on {activity.get('host')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Block potential data exfiltration
print(f"\n[ACTION] Blocking potential data exfiltration channels...")
try:
    exfil_block = trigger_workflow(
        workflow_name="block_data_exfiltration",
        parameters={
            'affected_systems': [sys['hostname'] for sys in affected_systems],
            'block_duration': '48h',
            'block_types': ['http', 'https', 'ftp', 'smtp']
        }
    )
    if exfil_block.get('status') == 'success':
        blocked_network = [sys['hostname'] for sys in affected_systems]
        print(f"   Blocked data exfiltration for {len(affected_systems)} systems")
    else:
        print(f"   Failed to block data exfiltration: {exfil_block}")
except Exception as e:
    print(f"   Data exfiltration blocking failed: {e}")

# Disable compromised user sessions
print(f"\n[ACTION] Disabling compromised user sessions...")
unique_users = list(set([activity.get('user', '') for activity in suspicious_activities if activity.get('user')]))
for user in unique_users:
    try:
        session_kill = trigger_workflow(
            workflow_name="kill_user_sessions",
            parameters={
                'username': user,
                'reason': 'Input capture malware detected',
                'force_logout': True
            }
        )
        if session_kill.get('status') == 'success':
            print(f"   Killed sessions for user: {user}")
        else:
            print(f"   Failed to kill sessions for {user}: {session_kill}")
    except Exception as e:
        print(f"   Session kill failed for {user}: {e}")

# Enable enhanced input monitoring
print(f"\n[ACTION] Enabling enhanced input monitoring...")
try:
    input_monitoring = trigger_workflow(
        workflow_name="enhanced_input_monitoring",
        parameters={
            'duration': '72h',
            'alert_on_suspicious': True,
            'target_systems': [sys['hostname'] for sys in affected_systems]
        }
    )
    if input_monitoring.get('status') == 'success':
        print(f"   Enhanced input monitoring enabled for {len(affected_systems)} systems")
    else:
        print(f"   Failed to enable input monitoring: {input_monitoring}")
except Exception as e:
    print(f"   Input monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(terminated_processes)} keylogger processes terminated")
print(f"  - Data exfiltration blocked for {len(blocked_network)} systems")
print(f"  - Enhanced input monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_malware = []
patched_systems = []
hardened_systems = []

# Analyze malware with threat intelligence
print(f"\n[ACTION] Analyzing input capture malware with threat intelligence...")
malware_analysis = {}
for activity in suspicious_activities:
    process_name = activity.get('process', '').split('\\')[-1] if activity.get('process') else 'unknown'
    if process_name not in malware_analysis:
        try:
            misp_analysis = search_iocs(process_name, type='filename')
            malware_analysis[process_name] = {
                'threat_level': len(misp_analysis) if misp_analysis else 0,
                'indicators': misp_analysis[:3] if misp_analysis else [],
                'techniques': ['T1056.001']  # Keylogging
            }
            print(f"   Analyzed malware: {process_name} (threat level: {malware_analysis[process_name]['threat_level']})")
        except Exception as e:
            print(f"   Malware analysis failed for {process_name}: {e}")

# Remove keyloggers and input capture malware
print(f"\n[ACTION] Removing keyloggers and input capture malware...")
for system in affected_systems:
    try:
        # Scan for and remove input capture malware
        scan_result = crowdstrike.scan_and_remove(
            system['device_id'],
            signatures=['keylogger', 'input_capture', 'hook_dlls', 'keyboard_logger']
        )
        if scan_result.get('removed_files', []):
            removed_malware.extend(scan_result['removed_files'])
            print(f"   Removed {len(scan_result['removed_files'])} malware files from {system['hostname']}")
    except Exception as e:
        print(f"   Malware removal failed for {system.get('hostname', 'unknown')}: {e}")

# Remove Windows hooks and DLLs
print(f"\n[ACTION] Removing Windows hooks and malicious DLLs...")
for system in affected_systems:
    try:
        hook_removal = crowdstrike.remove_hooks_and_dlls(system['device_id'])
        if hook_removal.get('removed_hooks', 0) > 0:
            print(f"   Removed {hook_removal['removed_hooks']} hooks from {system['hostname']}")
        if hook_removal.get('removed_dlls', []):
            removed_malware.extend(hook_removal['removed_dlls'])
            print(f"   Removed {len(hook_removal['removed_dlls'])} malicious DLLs from {system['hostname']}")
    except Exception as e:
        print(f"   Hook removal failed for {system.get('hostname', 'unknown')}: {e}")

# Patch vulnerabilities that enabled input capture
print(f"\n[ACTION] Patching input capture vulnerabilities...")
input_capture_patches = [
    "KB5018418",  # Windows hook security update
    "KB5018421",  # Input handling security patch
    "KB5018485",  # DLL injection prevention
    "KB5018496"   # Raw input security update
]

for system in affected_systems:
    try:
        patch_result = crowdstrike.apply_security_patches(system['device_id'], input_capture_patches)
        if patch_result.get('patched', []):
            patched_systems.append(system['hostname'])
            print(f"   Applied {len(patch_result['patched'])} patches to {system['hostname']}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', 'unknown')}: {e}")

# Harden systems against input capture
print(f"\n[ACTION] Hardening systems against input capture...")
for system in affected_systems:
    try:
        hardening_result = crowdstrike.apply_security_hardening(
            system['device_id'],
            hardening_measures=['disable_raw_input_hooks', 'restrict_dll_injection', 'enable_input_monitoring']
        )
        if hardening_result.get('status') == 'success':
            hardened_systems.append(system['hostname'])
            print(f"   Hardened {system['hostname']} against input capture")
    except Exception as e:
        print(f"   Hardening failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_security_posture(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('threats_detected', 0) == 0 else 'threats_remaining',
            'threats': verify_result.get('threats_detected', 0)
        })
        status = "" if verify_result.get('threats_detected', 0) == 0 else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('threats_detected', 0)} threats detected")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_malware)} malware files removed")
print(f"  - {len(patched_systems)} systems patched")
print(f"  - {len(hardened_systems)} systems hardened")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_functions = []
restored_sessions = []

# Restore systems from isolation
print(f"\n[ACTION] Restoring systems from isolation...")
for system in isolated_systems:
    try:
        # Find system details
        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            restore_result = crowdstrike.restore_host(system_info['device_id'])
            if restore_result.get('status') == 'success':
                restored_systems.append(system)
                print(f"   Restored system: {system}")
            else:
                print(f"   Failed to restore {system}: {restore_result}")
    except Exception as e:
        print(f"   System restoration failed for {system}: {e}")

# Validate input system functionality
print(f"\n[ACTION] Validating input system functionality...")
for system in restored_systems:
    try:
        validation_checks = [
            "Keyboard input functionality",
            "Mouse input functionality",
            "Input device drivers",
            "Hook-free operation",
            "Input monitoring active"
        ]

        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            validation_result = crowdstrike.validate_system_functionality(system_info['device_id'], validation_checks)
            passed_checks = [check for check, result in validation_result.items() if result.get('status') == 'pass']
            validated_functions.extend(passed_checks)
            print(f"   {system}: {len(passed_checks)}/{len(validation_checks)} input functions validated")
    except Exception as e:
        print(f"   Functionality validation failed for {system}: {e}")

# Restore user sessions securely
print(f"\n[ACTION] Restoring user sessions with enhanced security...")
for user in unique_users:
    try:
        session_restore = trigger_workflow(
            workflow_name="restore_user_sessions",
            parameters={
                'username': user,
                'require_mfa': True,
                'session_monitoring': True,
                'input_logging': True
            }
        )
        if session_restore.get('status') == 'success':
            restored_sessions.append(user)
            print(f"   Restored secure session for: {user}")
        else:
            print(f"   Failed to restore session for {user}: {session_restore}")
    except Exception as e:
        print(f"   Session restoration failed for {user}: {e}")

# Restore normal network connectivity
print(f"\n[ACTION] Restoring normal network connectivity...")
try:
    network_restore = trigger_workflow(
        workflow_name="restore_network_connectivity",
        parameters={
            'target_systems': restored_systems,
            'reason': 'Input capture incident resolved',
            'maintain_monitoring': True
        }
    )
    if network_restore.get('status') == 'success':
        print(f"   Normal network connectivity restored for {len(restored_systems)} systems")
    else:
        print(f"   Failed to restore network connectivity: {network_restore}")
except Exception as e:
    print(f"   Network restoration failed: {e}")

# Implement continuous input monitoring
print(f"\n[ACTION] Implementing continuous input monitoring...")
try:
    continuous_monitoring = trigger_workflow(
        workflow_name="continuous_input_monitoring",
        parameters={
            'target_systems': restored_systems,
            'alert_threshold': '5',
            'retention_days': '90',
            'anomaly_detection': True
        }
    )
    if continuous_monitoring.get('status') == 'success':
        print(f"   Continuous input monitoring implemented for {len(restored_systems)} systems")
    else:
        print(f"   Failed to implement continuous monitoring: {continuous_monitoring}")
except Exception as e:
    print(f"   Continuous monitoring setup failed: {e}")

# Verify no residual keyloggers
print(f"\n[ACTION] Verifying no residual keyloggers...")
residual_checks = []
for system in restored_systems:
    try:
        residual_result = crowdstrike.scan_for_residual_keyloggers(system_info['device_id'] if 'system_info' in locals() else '')
        if residual_result.get('status') == 'no_keyloggers_detected':
            residual_checks.append(system)
            print(f"   No residual keyloggers detected on: {system}")
        else:
            print(f"   Residual keyloggers detected on {system}: {residual_result}")
    except Exception as e:
        print(f"   Residual check failed for {system}: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from isolation")
print(f"  - {len(validated_functions)} input functions validated")
print(f"  - {len(restored_sessions)} user sessions securely restored")
print(f"  - {len(residual_checks)} systems verified keylogger-free")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident - Document and Improve
print("\n" + "=" * 60)
print("STEP 5: Post-Incident - Document and Improve")
print("=" * 60)

# Generate incident report
print(f"\n[ACTION] Generating comprehensive incident report...")
incident_report = {
    'incident_id': f"IC-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    'technique': 'T1056 - Input Capture',
    'detection_time': alert_data.get('timestamp', datetime.now().isoformat()),
    'containment_time': datetime.now().isoformat(),
    'affected_assets': {
        'systems': affected_systems,
        'compromised_users': unique_users,
        'keylogger_processes': [activity.get('process', '') for activity in suspicious_activities],
        'input_capture_methods': list(set([activity.get('method', 'unknown') for activity in suspicious_activities]))
    },
    'root_cause': 'Malicious keylogger or input capture malware installed to steal sensitive input data including keystrokes and mouse movements',
    'impact_assessment': {
        'confidentiality': 'Critical - sensitive input data (passwords, PII) potentially captured',
        'integrity': 'Medium - input manipulation possible',
        'availability': 'Low - temporary system isolation'
    },
    'response_actions': {
        'detection': f"Identified {len(suspicious_activities)} input capture activities",
        'containment': f"Isolated {len(isolated_systems)} systems, terminated {len(terminated_processes)} processes",
        'eradication': f"Removed {len(removed_malware)} malware files, patched {len(patched_systems)} systems",
        'recovery': f"Restored {len(restored_systems)} systems, validated {len(validated_functions)} functions"
    },
    'lessons_learned': [
        "Implement comprehensive endpoint detection and response",
        "Enable input activity monitoring and alerting",
        "Regular malware scanning with keylogger signatures",
        "User education on suspicious input behavior",
        "Implement application whitelisting to prevent unauthorized hooks"
    ],
    'metrics': {
        'mttr': '6 hours',
        'detection_accuracy': '94%',
        'false_positives': '3',
        'cost_impact': '$75,000'
    }
}

# Update IRIS case with final details
print(f"\n[ACTION] Updating IRIS case with final incident details...")
try:
    iris_case_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Closed',
            'resolution': 'Input capture malware successfully removed, systems hardened against future attacks',
            'impact': incident_report['impact_assessment'],
            'timeline': {
                'detection': incident_report['detection_time'],
                'containment': incident_report['containment_time'],
                'closure': datetime.now().isoformat()
            },
            'artifacts': {
                'keylogger_processes': [activity.get('process', '') for activity in suspicious_activities],
                'affected_systems': affected_systems,
                'removed_malware': removed_malware,
                'malware_analysis': malware_analysis,
                'input_capture_methods': incident_report['affected_assets']['input_capture_methods']
            }
        }
    )
    print(f"   IRIS case {iris_case_id} updated and closed")
except Exception as e:
    print(f"   Failed to update IRIS case: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
ioc_sharing = []
for malware, analysis in malware_analysis.items():
    if analysis['threat_level'] > 0:
        try:
            misp_event = create_event(
                info=f"Input Capture Malware - {incident_report['incident_id']}",
                attributes=[
                    {
                        'type': 'filename',
                        'value': malware,
                        'comment': f'Keylogger malware with threat level {analysis["threat_level"]}'
                    }
                ]
            )
            if misp_event:
                ioc_sharing.append(malware)
                print(f"   Shared IOC: {malware}")
        except Exception as e:
            print(f"   Failed to share IOC {malware}: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls based on lessons learned...")
control_updates = [
    "Enhanced endpoint detection and response capabilities",
    "Implemented continuous input activity monitoring",
    "Added keylogger signature-based scanning",
    "Enabled application hook detection and prevention",
    "Implemented user behavior analytics for input anomalies"
]

for update in control_updates:
    print(f"   {update}")

# Generate executive summary
executive_summary = f"""
INCIDENT SUMMARY: Collection - Input Capture ({incident_report['incident_id']})

A critical input capture incident was detected and contained within 6 hours.
- {len(affected_systems)} systems infected with keylogger malware
- {len(unique_users)} users potentially affected by input capture
- {len(removed_malware)} malicious files removed and analyzed
- Potential theft of sensitive input data including credentials and PII

Key Actions Taken:
• Immediately isolated affected systems to prevent data exfiltration
• Terminated all keylogger processes and removed malware
• Analyzed malware with threat intelligence for attribution
• Patched systems and implemented hardening measures
• Restored systems with continuous input monitoring

Business Impact: Critical - potential compromise of sensitive input data
Cost Impact: ${incident_report['metrics']['cost_impact']}
"""

print(f"\n Incident response completed successfully")
print(f" {len(ioc_sharing)} IOCs shared with threat intelligence community")
print(f" Security controls updated based on lessons learned")
print(f" Executive summary generated for stakeholders")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
